# 05 — Forecasting Models

One **global** LightGBM across all series, not one model per SKU: most series are too sparse to support their own model, new products have no history at all, and per-series fitting costs scale with the number of series for no additional insight.

Objective is Tweedie because demand is a non-negative, zero-inflated count; squared error treats it as symmetric and unbounded.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore')

from src.config import load_config
from src.data.loader import load_panel
cfg = load_config()
panel = load_panel(cfg)
print(f'panel: {len(panel):,} rows, {panel.series_id.nunique()} series, {panel.date.nunique()} days')

## Walk-forward validation with refitting per fold

Runs several minutes: the model is refitted for each fold, because reusing one model fitted on all training data would let fold 1 be scored by a model that had seen fold 4.

In [ ]:
from src.evaluation.validate import walk_forward, aggregate_scores, per_fold_table
from src.evaluation.metrics import skill_vs
res = walk_forward(cfg, panel, fit_ml=True, n_origins=20)
skill_vs(aggregate_scores(res), 'seasonal_naive').round(4)

In [ ]:
per_fold_table(res).round(4)

## Feature importance

In [ ]:
from src.features.build import build_supervised, training_origins
from src.forecasting import ml as ML
from src.data.loader import rolling_origin_folds
fold = rolling_origin_folds(cfg, panel)[-1]
tr = build_supervised(cfg, panel, fold,
                      origins=training_origins(cfg, panel, fold.origin, 20))
m = ML.fit_lightgbm(cfg, tr)
ML.feature_importance(m, top=15)

## Classical models on a stratified sample

ETS and SARIMA are fitted per series, so only on a sample. Their assumptions (a continuous, roughly Gaussian process) fail on intermittent series, which is a result worth reporting rather than a bug to hide.

In [ ]:
from src.forecasting.classical import forecast_classical, sample_series
sample = sample_series(cfg, panel)
cl = forecast_classical(cfg, panel, fold, sample)
print('fit failures:', cl.n_failed)
for n in cl.notes: print('note:', n)